## Per-step NCS from the raw archive

Recomputes the core measurement straight from `temp/probe_output/`, independent of
`ncs_long.csv`. For one distillation event it builds the teacher-side bin masks,
loads the student's logits before and after, and returns corrections / corruptions /
NCS plus the three accuracies per bin.

Two things that are easy to get wrong and are handled explicitly below:

- **The binning axis is the *teacher's*.** E-step (`gnn->lm`) bins on local homophily,
  low = out-of-bias; M-step (`lm->gnn`) bins on kNN ambiguity, high = out-of-bias.
  Binning on the student's axis answers a different question.
- **"Before" is not the previous step.** Consecutive step indices alternate between the
  LM and the GNN, so `step i` vs `step i+1` compares two *different models*. The
  student's pre-step state is the same model's snapshot from the previous iteration
  (`iter{i-1}`), which is what `step_logits` returns.

Verified against `ncs_long.csv`: 156 bins, 0 mismatches.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve the repo root from this file's location rather than the kernel's cwd, so
# the notebook works whether it is started from the repo root or from notebooks/.
ROOT = Path.cwd()
while not (ROOT / 'src' / 'probe').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
PROBE = ROOT / 'temp' / 'probe_output'
SIGNALS = PROBE / '_signals'

# Per direction: which signal bins the nodes, and which side is the region where the
# TEACHER is outside its inductive bias. Getting this backwards silently answers a
# different question ("does transfer help where the STUDENT is weak?"), so it is
# written out once here and never re-derived.
AXIS = {
    'gnn->lm': ('local_homophily', 'low'),   # GNN teaches the LM; GNN fails at LOW homophily
    'lm->gnn': ('knn_ambiguity',  'high'),   # LM teaches the GNN; LM fails at HIGH ambiguity
}


def load_run(dataset, arm='published', seed=0, regime='standard'):
    """(run_dir, steps, splits) for one cell. `dataset` is e.g. 'cornell_TAG'."""
    run = PROBE / dataset / regime / arm / f'seed{seed}'
    steps = [json.loads(l) for l in (run / 'steps.jsonl').read_text().splitlines() if l.strip()]
    return run, steps, np.load(run / 'splits.npz')


def signal_values(dataset, direction, seed=0, regime='standard'):
    """The per-node signal array for this direction's teacher-side axis."""
    key = dataset.split('_')[0]
    name, _ = AXIS[direction]
    if name == 'local_homophily':
        return np.load(SIGNALS / f'{key}_homophily.npy'), name
    return np.load(SIGNALS / f'{key}_{regime}_s{seed}_ambiguity.npy'), name


def population_for_step(run, step, splits):
    """Nodes this step actually taught AND for which we have ground truth.

    Both conditions matter: `plnodes` is what the run logged at the time (the GNN
    resamples every epoch, so it is not recoverable afterwards), and ground truth
    transductively means val u test.
    """
    pl = np.load(run / 'plnodes' / f"step{step['step_index']}_{step['em_phase']}.npy")
    return np.intersect1d(pl, np.union1d(splits['valid_x'], splits['test_x']))


def bin_masks(dataset, direction, population, seed=0, regime='standard', scheme='median'):
    """Split `population` into the teacher's in-bias and out-of-bias node ids.

    The cut is taken over the analysed population, not over all nodes, so the two
    cells stay balanced in the group actually being tested. Non-finite values
    (isolated nodes have NaN local homophily) are dropped before the cut so they
    cannot shift it. Returns {} if either side is empty -- a one-sided split is a
    missing comparison, not a result.
    """
    vals, name = signal_values(dataset, direction, seed, regime)
    v = vals[population]
    keep = np.isfinite(v)
    ids, v = np.asarray(population)[keep], v[keep]
    if len(ids) == 0:
        return {}
    cut = float(np.median(v)) if scheme == 'median' else float(v.mean())
    low, high = ids[v <= cut], ids[v > cut]          # ties go low, as in the pipeline
    if len(low) == 0 or len(high) == 0:
        return {}
    _, oob_side = AXIS[direction]
    oob, inb = (low, high) if oob_side == 'low' else (high, low)
    return {'out_of_bias': oob, 'in_bias': inb, 'signal': name,
            'cut': cut, 'scheme': scheme, 'n_dropped_nan': int((~keep).sum())}


def step_logits(run, step):
    """(before, after, teacher) logits for one distillation event, as float32.

    `after` is the student's snapshot for this iteration; `before` is the SAME
    model's snapshot from the previous iteration (`iter-1` is the pretrained
    baseline). Note this is not "step i vs step i+1": consecutive step indices
    alternate between the LM and the GNN, so comparing them would compare two
    different models rather than one model before and after being taught.
    """
    k = 'lm' if step['student'] == 'LM' else 'gnn'
    i = step['em_iter']
    load = lambda p: np.load(p).astype(np.float32)
    return (load(run / 'logits' / f'iter{i - 1}_{k}.npy'),
            load(run / 'logits' / f'iter{i}_{k}.npy'),
            load(run / 'teacher' / f"step{step['step_index']}_{step['em_phase']}.npy"))


def bin_stats(before, after, teacher, labels, ids):
    """Corrections / corruptions / NCS and the three accuracies for one bin."""
    ok_b = before[ids].argmax(1) == labels[ids]
    ok_a = after[ids].argmax(1) == labels[ids]
    corrections = int((~ok_b & ok_a).sum())     # wrong  -> correct
    corruptions = int((ok_b & ~ok_a).sum())     # correct -> wrong
    n = len(ids)
    return {'n': n, 'corrections': corrections, 'corruptions': corruptions,
            'ncs': (corrections - corruptions) / n,
            'corruption_rate': corruptions / n,
            'student_acc_before': float(ok_b.mean()),
            'student_acc_after': float(ok_a.mean()),
            'teacher_acc': float((teacher[ids].argmax(1) == labels[ids]).mean())}


def analyse_step(dataset, step_index, arm='published', seed=0, regime='standard',
                 scheme='median'):
    """One row per bin for a single distillation event."""
    run, steps, splits = load_run(dataset, arm, seed, regime)
    step = next(s for s in steps if s['step_index'] == step_index)
    labels = splits['labels']
    pop = population_for_step(run, step, splits)
    bins = bin_masks(dataset, step['direction'], pop, seed, regime, scheme)
    if not bins:
        return pd.DataFrame()
    before, after, teacher = step_logits(run, step)
    rows = []
    for role in ('in_bias', 'out_of_bias'):
        rows.append({'dataset': dataset.split('_')[0], 'arm': arm, 'seed': seed,
                     'step': step_index, 'iteration': step['em_iter'],
                     'direction': step['direction'], 'teacher': step['teacher'],
                     'signal': bins['signal'], 'cut': round(bins['cut'], 4),
                     'role': role, **bin_stats(before, after, teacher, labels, bins[role])})
    return pd.DataFrame(rows)


def analyse_dataset(dataset, arm='published', seed=0, regime='standard', scheme='median'):
    """Every distillation event in one run."""
    _, steps, _ = load_run(dataset, arm, seed, regime)
    out = [analyse_step(dataset, s['step_index'], arm, seed, regime, scheme) for s in steps]
    out = [o for o in out if not o.empty]
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()

### Usage

```python
analyse_dataset('cornell_TAG')                      # every step of one run
analyse_step('arxiv_TA', step_index=0)              # a single distillation event
analyse_dataset('cora_TAG', arm='alpha0_li_T')      # the alpha=beta=0 control
analyse_dataset('cora_TAG', scheme='mean')          # exploratory mean split (A11)
```

`analyse_step` returns an empty frame where the split degenerates — on cora, citeseer
and pubmed the E-step median homophily is 1.000, so the high bin is empty and there is
no comparison to make (amendment A6). Try `scheme='mean'` there.

Available datasets: `arxiv_TA`, `cora_TAG`, `citeseer_TAG`, `pubmed_TAG`,
`cornell_TAG`, `texas_TAG`, `washington_TAG`, `wisconsin_TAG`.
Arms: `published`, `alpha0_li_T`, and `alpha0_li_F` on the four WebKB sets.
Seeds 0-2, except arxiv which has seed 0 only.

In [ ]:
datasets = ['cornell_TAG', 'texas_TAG', 'wisconsin_TAG', 'washington_TAG', 'cora_TAG', 'citeseer_TAG', 'pubmed_TAG', 'arxiv_TA']
for dataset in datasets:
        print(f"=== {dataset} ===")
        res = analyse_dataset(dataset, arm='published', seed=0, scheme='mean')
        pd.set_option('display.width', 220)
        print(res.round(4).to_string(index=False))
        # The comparison the hypothesis is about: out-of-bias vs in-bias, pooled over steps.
        print()
        print(res.groupby(['direction', 'role'])[['n', 'corrections', 'corruptions']].sum()
                .assign(ncs=lambda d: (d.corrections - d.corruptions) / d.n).round(4).to_string())

=== cornell_TAG ===


FileNotFoundError: [Errno 2] No such file or directory: 'temp/probe_output/cornell_TAG/standard/published/seed0/steps.jsonl'

In [ ]:
def fast_auroc(scores, labels):
    """Rank-based AUROC (Mann-Whitney U), tie-corrected. NaN if one class absent."""
    import numpy as np
    labels = np.asarray(labels).astype(bool)
    n_pos, n_neg = int(labels.sum()), int((~labels).sum())
    if n_pos == 0 or n_neg == 0:
        return float('nan')
    order = np.argsort(scores, kind='mergesort')
    ranks = np.empty(len(scores), dtype=float)
    ranks[order] = np.arange(1, len(scores) + 1)
    s = scores[order]
    i = 0
    while i < len(s):                      # average ranks within tied groups
        j = i + 1
        while j < len(s) and s[j] == s[i]:
            j += 1
        if j - i > 1:
            ranks[order[i:j]] = (i + 1 + j) / 2.0
        i = j
    return float((ranks[labels].sum() - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg))


def confidence_blindness(dataset, arm='published', seed=0, regime='standard', scheme='median'):
    """Does the teacher's own confidence predict whether the teacher is right?

    GLEM's only per-node mechanism is `pl_filter`, which keeps the top fraction by
    max-softmax. The hypothesis behind this experiment says that outside its
    inductive bias a model is not uncertain but *confidently wrong* -- which
    predicts confidence should stop discriminating correctness precisely in the
    out-of-bias region.

    AUROC ~ 0.5 means confidence carries no information about correctness there,
    i.e. a confidence gate cannot help. Reported per bin so in-bias is the control.
    """
    import numpy as np, pandas as pd
    run, steps, splits = load_run(dataset, arm, seed, regime)
    labels = splits['labels']
    rows = []
    for st in steps:
        pop = population_for_step(run, st, splits)
        bins = bin_masks(dataset, st['direction'], pop, seed, regime, scheme)
        if not bins:
            continue
        _, _, teacher = step_logits(run, st)
        p = np.exp(teacher - teacher.max(1, keepdims=True))
        p /= p.sum(1, keepdims=True)
        conf, pred = p.max(1), p.argmax(1)
        for role in ('in_bias', 'out_of_bias'):
            ids = bins[role]
            correct = pred[ids] == labels[ids]
            rows.append({'dataset': dataset.split('_')[0], 'arm': arm, 'seed': seed,
                         'step': st['step_index'], 'direction': st['direction'],
                         'teacher': st['teacher'], 'role': role, 'n': len(ids),
                         'teacher_acc': float(correct.mean()),
                         'mean_conf': float(conf[ids].mean()),
                         'auroc_conf_vs_correct': fast_auroc(conf[ids], correct)})
    return pd.DataFrame(rows)


def follow_teacher(dataset, arm='published', seed=0, regime='standard', scheme='median'):
    """Did the student adopt the teacher's *specific* wrong label?

    NCS nets corrections against corruptions and so hides the mechanism. This keys
    on the teacher's actual prediction instead: restrict to nodes the student had
    RIGHT and the teacher had WRONG -- the only nodes where following the teacher
    can do damage -- and ask how often the student ended up on the teacher's label.

    `follow_rate` is that fraction among the nodes that actually broke. Run it on
    the alpha=beta=0 control for the null: there the teacher's label has no causal
    path to the student, so any agreement is coincidence.
    """
    import numpy as np, pandas as pd
    run, steps, splits = load_run(dataset, arm, seed, regime)
    labels = splits['labels']
    rows = []
    for st in steps:
        pop = population_for_step(run, st, splits)
        bins = bin_masks(dataset, st['direction'], pop, seed, regime, scheme)
        if not bins:
            continue
        before, after, teacher = step_logits(run, st)
        t_pred, b_pred, a_pred = teacher.argmax(1), before.argmax(1), after.argmax(1)
        for role in ('in_bias', 'out_of_bias'):
            ids = bins[role]
            # nodes where following the teacher would break a correct prediction
            at_risk = ids[(b_pred[ids] == labels[ids]) & (t_pred[ids] != labels[ids])]
            broke = at_risk[a_pred[at_risk] != labels[at_risk]]
            to_teacher = broke[a_pred[broke] == t_pred[broke]]
            n_cls = int(labels.max()) + 1
            rows.append({'dataset': dataset.split('_')[0], 'arm': arm, 'seed': seed,
                         'step': st['step_index'], 'direction': st['direction'],
                         'role': role, 'n_bin': len(ids),
                         'at_risk': len(at_risk),          # student right, teacher wrong
                         'broke': len(broke),              # ...and student then broke
                         'to_teacher_label': len(to_teacher),
                         'break_rate': len(broke) / len(at_risk) if len(at_risk) else float('nan'),
                         'follow_rate': len(to_teacher) / len(broke) if len(broke) else float('nan'),
                         'chance': 1.0 / (n_cls - 1)})     # if it broke to a random wrong class
    return pd.DataFrame(rows)

In [ ]:
datasets = ['cornell_TAG', 'texas_TAG', 'wisconsin_TAG', 'washington_TAG', 'cora_TAG', 'citeseer_TAG', 'pubmed_TAG', 'arxiv_TA']
for dataset in datasets:
        print(f"=== {dataset} ===")
        res = analyse_dataset(dataset, arm='alpha0_li_T', seed=0, scheme='mean')
        pd.set_option('display.width', 220)
        print(res.round(4).to_string(index=False))
        # The comparison the hypothesis is about: out-of-bias vs in-bias, pooled over steps.
        print()
        print(res.groupby(['direction', 'role'])[['n', 'corrections', 'corruptions']].sum()
                .assign(ncs=lambda d: (d.corrections - d.corruptions) / d.n).round(4).to_string())

=== cornell_TAG ===
dataset         arm  seed  step  iteration direction teacher          signal    cut        role  n  corrections  corruptions     ncs  corruption_rate  student_acc_before  student_acc_after  teacher_acc
cornell alpha0_li_T     0     0          0   lm->gnn      LM   knn_ambiguity 0.6409     in_bias 25            4            3  0.0400           0.1200              0.7200             0.7600       0.7200
cornell alpha0_li_T     0     0          0   lm->gnn      LM   knn_ambiguity 0.6409 out_of_bias 40            4            6 -0.0500           0.1500              0.6000             0.5500       0.4250
cornell alpha0_li_T     0     1          0   gnn->lm     GNN local_homophily 0.1611     in_bias  4            0            0  0.0000           0.0000              0.5000             0.5000       0.5000
cornell alpha0_li_T     0     1          0   gnn->lm     GNN local_homophily 0.1611 out_of_bias  5            1            0  0.2000           0.0000              0.6000   

## Two diagnostics NCS cannot give you

NCS answers "did this bin get net better or worse". It cannot say **why**, because it
nets corrections against corruptions and never looks at *which* label the student moved
to. These two tests key on the teacher's actual predictions instead. Both run off the
existing archive — no retraining.

**1. Confidence blindness.** GLEM's only per-node mechanism is `pl_filter`, which keeps
the top fraction of nodes by max-softmax confidence. The premise of this experiment is
that outside its inductive bias a model is not *uncertain* but *confidently wrong* — which
predicts that confidence should stop discriminating correctness precisely out-of-bias.
AUROC near 0.5 means a confidence gate cannot help there.

**2. Following the teacher.** Restrict to nodes the student had **right** and the teacher
had **wrong** — the only nodes where obeying the teacher can do damage — then ask how
often the student ended on the teacher's *specific* label. Run it on the α=β=0 control
for the null: there the teacher's label has no causal path to the student, so agreement
is coincidence.

### Reading the output

```python
confidence_blindness('arxiv_TA')                      # per step and bin
follow_teacher('arxiv_TA', arm='published')           # then again with arm='alpha0_li_T'
```

On arxiv both come out strongly:

| | in-bias | out-of-bias |
|---|---|---|
| teacher accuracy (`gnn->lm`) | 0.957 | 0.581 |
| mean max-softmax confidence | 0.940 | 0.795 |
| **AUROC, confidence vs correctness** | **0.925** | **0.628** |

The teacher stays highly confident where it is only 58% accurate, and its confidence
loses most of its power to tell right from wrong — 0.925 → 0.628. That is "confidently
wrong" quantified, and it is direct evidence that a `pl_filter`-style confidence gate is
weakest exactly where it would need to be strongest.

For test 2, `gnn->lm` out-of-bias, comparing arms:

| arm | at-risk | break rate | follow rate |
|---|---|---|---|
| published | 4,039 | **0.537** | **0.902** |
| α=β=0 control | 5,239 | 0.216 | 0.674 |

With the pseudo-label term on, an at-risk node breaks **2.5× more often**, and when it
breaks it lands on the teacher's exact label 90% of the time — against a 2.6% chance
baseline (1/39 wrong classes). The student is plainly following the teacher.

**Two cautions.** The control's follow rate is also high (0.674), because teacher and
student are trained on the same data and confuse the same classes — agreement is not
causation, which is why the arm comparison rather than the raw rate is the evidence. And
the amplification is similar in-bias (3.8×) and out-of-bias (3.3×): the student follows
the teacher everywhere, so the harm concentrates out-of-bias simply because the teacher
is *wrong* more often there (4,039 at-risk nodes vs 353), not because following is more
likely there. That, together with the teacher still being right on many nodes the student
had wrong, is why NCS stays positive despite this mechanism being real.

In [ ]:
DATASETS = ['cornell_TAG', 'texas_TAG', 'washington_TAG', 'wisconsin_TAG',
            'cora_TAG', 'citeseer_TAG', 'pubmed_TAG', 'arxiv_TA']
SCHEME = 'median'      # 'mean' also bins the E-step on cora/citeseer/pubmed (A6)
MIN_N = 30             # same floor the preregistered analysis uses

# ---- Test 1: does the teacher's confidence predict its own correctness? ----
conf = pd.concat([confidence_blindness(ds, arm='published', seed=0, scheme=SCHEME)
                  for ds in DATASETS], ignore_index=True)
conf = conf[conf.n >= MIN_N]
t1 = (conf.groupby(['direction', 'dataset', 'role'])
          .agg(n=('n', 'sum'), teacher_acc=('teacher_acc', 'mean'),
               mean_conf=('mean_conf', 'mean'), auroc=('auroc_conf_vs_correct', 'mean'))
          .unstack('role'))
t1['auroc_drop'] = t1[('auroc', 'in_bias')] - t1[('auroc', 'out_of_bias')]
pd.set_option('display.width', 210)
print('TEST 1 — AUROC of teacher confidence vs teacher correctness (published, seed 0)')
print('a drop toward 0.5 out-of-bias = a confidence gate cannot see the teacher failing')
print(t1[['teacher_acc', 'auroc', 'auroc_drop']].round(3).to_string())

# ---- Test 2: does the student adopt the teacher's specific wrong label? ----
foll = pd.concat([follow_teacher(ds, arm=a, seed=0, scheme=SCHEME)
                  for ds in DATASETS for a in ('published', 'alpha0_li_T')],
                 ignore_index=True)
g = (foll.groupby(['direction', 'dataset', 'arm', 'role'])
         .agg(at_risk=('at_risk', 'sum'), broke=('broke', 'sum'),
              to_teacher=('to_teacher_label', 'sum'), chance=('chance', 'mean'))
         .reset_index())
g = g[g.at_risk >= MIN_N]
g['break_rate'] = g.broke / g.at_risk
g['follow_rate'] = g.to_teacher / g.broke
# P(node breaks AND lands on the teacher's label) -- the quantity the arms should differ on
g['break_and_follow'] = g.to_teacher / g.at_risk
print()
print('TEST 2 — among nodes the student had RIGHT and the teacher had WRONG (seed 0)')
print('chance of matching the teacher by accident = 1/(C-1), the `chance` column')
print(g[['direction', 'dataset', 'arm', 'role', 'at_risk', 'break_rate',
         'follow_rate', 'break_and_follow', 'chance']].round(3).to_string(index=False))

# ---- the arm comparison, which is the actual evidence ----
piv = g.pivot_table(index=['direction', 'dataset', 'role'], columns='arm',
                    values='break_and_follow')
if {'published', 'alpha0_li_T'} <= set(piv.columns):
    piv['amplification'] = piv['published'] / piv['alpha0_li_T']
    print()
    print('P(breaks AND lands on teacher label): published vs control, and their ratio')
    print('>1 means the pseudo-label term is causing the student to adopt wrong labels')
    print(piv.round(3).to_string())

TEST 1 — AUROC of teacher confidence vs teacher correctness (published, seed 0)
a drop toward 0.5 out-of-bias = a confidence gate cannot see the teacher failing
                     teacher_acc               auroc             auroc_drop
role                     in_bias out_of_bias in_bias out_of_bias           
direction dataset                                                          
gnn->lm   arxiv            0.957       0.581   0.925       0.627      0.298
lm->gnn   arxiv            0.876       0.627   0.769       0.688      0.082
          citeseer         0.258       0.189   0.558       0.571     -0.013
          cora             0.957       0.706   0.857       0.718      0.139
          cornell          0.380       0.250   0.570       0.667     -0.096
          pubmed           0.977       0.914   0.903       0.882      0.021
          texas            0.774         NaN   0.756         NaN        NaN
          washington       0.775       0.554   0.582       0.652     -0.070
   

### What the two tables say

**Test 1 — confidence goes blind where it matters.** On arxiv's E-step, AUROC falls
0.925 → 0.628 as the GNN teacher drops from 96% to 58% accurate, while its mean
confidence only falls 0.94 → 0.80. It stays sure and stops being right. cora shows the
same shape (0.857 → 0.718). Three datasets go the *other* way — citeseer −0.013, cornell
−0.096, washington −0.070 — but those are the small or degenerate-teacher cases, and
citeseer's teacher is at chance in both bins so its AUROC means little either way.

**Test 2 — the student really does adopt the teacher's wrong label.** The last table is
the evidence: P(an at-risk node breaks *and* lands on the teacher's exact label),
published against the α=β=0 control. Every cell is **1.7×–5.8×** higher with the
pseudo-label term on. The mechanism the hypothesis proposed is real and large.

**But it is not concentrated out-of-bias.** Amplification is 3.79× in-bias vs 3.33×
out-of-bias on arxiv's E-step, 2.07× vs 1.87× on its M-step — flat, or slightly *lower*
where the teacher is weak. Only citeseer inverts it (4.90× vs 5.76×). The student follows
the teacher uniformly; what changes across the axis is how often the teacher is wrong
(4,039 at-risk nodes out-of-bias vs 353 in-bias on arxiv).

So the corrected picture: **the harm mechanism is confirmed but is not bias-dependent.**
Damage concentrates out-of-bias purely through the volume of teacher errors, and stays
net-positive because the teacher is simultaneously right on many nodes the student had
wrong. That is a different claim from the original hypothesis — and a more useful one,
since it says a per-node gate should estimate *P(teacher wrong)*, which Test 1 shows the
teacher's own confidence cannot do where it counts.

## Perfect-teacher (oracle) sweep — how much is per-node gating worth?

The preregistered study found no *harm* from uniform α: the teacher stays better than
the student even where it is weakest. This sweep asks the complementary question —
**if the teacher's wrong labels could be removed perfectly, how much accuracy would
that buy?** That bounds every realisable gate, including GLEM's own `pl_filter` and
any per-node α predicted from exogenous signals.

Two arms, 7 RevGAT datasets × 3 seeds (EXPERIMENT.md A14):

- `oracle` — pseudo-label set restricted to nodes the teacher gets **right**
- `oracle_random` — drops the same *number* at random

**Compare oracle against `oracle_random`, never against `published`.** The oracle
also shrinks the pseudo-label set, and shrinking it changes the result on its own;
the `shrink_cost` column below measures that separately.

The second table is the one that matters, and it is there because the arm has a flaw
(A15). The oracle selects among the unlabeled set, which transductively **is**
val ∪ test — so every node it keeps is trained with its own correct label and then
evaluated on. That is memorisable. `not_kept_gain` isolates the part of the gain that
is not leaked supervision.

In [ ]:
import json
from pathlib import Path
import numpy as np, pandas as pd

ROOT = Path.cwd()
while not (ROOT / 'src' / 'probe').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
PROBE = ROOT / 'temp' / 'probe_output'

def final_accuracy(probe=PROBE):
    """Test/val accuracy of each model at the last EM iteration, per run."""
    rows = []
    for run in sorted(probe.glob('*/standard/*/seed*')):
        sp = run / 'splits.npz'
        if not sp.exists():
            continue
        z = np.load(sp); y = z['labels']
        for kind in ('gnn', 'lm'):
            f = run / 'logits' / ('iter1_%s.npy' % kind)
            if not f.exists():
                continue
            pred = np.load(f).astype(np.float32).argmax(1)
            rows.append({'dataset': run.parts[-4], 'arm': run.parts[-2],
                         'seed': int(run.name[4:]), 'model': kind,
                         'test_acc': float((pred[z['test_x']] == y[z['test_x']]).mean()),
                         'val_acc': float((pred[z['valid_x']] == y[z['valid_x']]).mean())})
    return pd.DataFrame(rows)

def oracle_effect(acc):
    """Paired oracle - oracle_random per (dataset, model, seed).

    A14: the size-matched random arm is the comparator, never `published` --
    the oracle also shrinks the pseudo-label set, and shrinking it changes the
    result on its own.
    """
    w = acc.pivot_table(index=['dataset', 'model', 'seed'], columns='arm',
                        values='test_acc')
    need = {'oracle', 'oracle_random', 'published'}
    if not need <= set(w.columns):
        return pd.DataFrame()
    w = w.dropna(subset=['oracle', 'oracle_random'])
    w['oracle_vs_random'] = w['oracle'] - w['oracle_random']
    w['random_vs_published'] = w['oracle_random'] - w['published']
    g = w.groupby(['dataset', 'model']).agg(
        seeds=('oracle', 'size'),
        oracle=('oracle', 'mean'), random=('oracle_random', 'mean'),
        published=('published', 'mean'),
        gain=('oracle_vs_random', 'mean'), gain_sd=('oracle_vs_random', 'std'),
        shrink_cost=('random_vs_published', 'mean'))
    g['gain_sign_stable'] = w.groupby(['dataset', 'model']).oracle_vs_random.apply(
        lambda s: bool(len(set(np.sign(s[s != 0]))) <= 1 and len(s) > 1))
    return g.reset_index()

def leakage_decomposition(probe=PROBE, seeds=(0, 1, 2)):
    """Split the oracle's test-set gain into kept vs not-kept nodes.

    The oracle selects among val u test using gold labels, so every node it keeps
    is trained with its CORRECT label -- and those same nodes are then evaluated.
    A transductive model can memorise them. Accuracy on the nodes the oracle did
    NOT keep is the only part of the gain that reflects better learning rather
    than leaked supervision.
    """
    rows = []
    for run in sorted(probe.glob('*/standard/oracle/seed*')):
        ds, seed = run.parts[-4], int(run.name[4:])
        if seed not in seeds:
            continue
        ctrl = run.parent.parent / 'oracle_random' / run.name
        if not (ctrl / 'logits' / 'iter1_gnn.npy').exists():
            continue
        z = np.load(run / 'splits.npz'); y = z['labels']; test = z['test_x']
        steps = [json.loads(l) for l in (run / 'steps.jsonl').read_text().splitlines()]
        for kind, phase in (('gnn', 'GNN'), ('lm', 'LM')):
            st = [s for s in steps if s['em_phase'] == phase]
            if not st:
                continue
            st = st[-1]
            kept = np.load(run / 'plnodes' / ('step%d_%s.npy' % (st['step_index'], phase)))
            inn = np.intersect1d(test, kept)            # trained on a correct label
            out = np.setdiff1d(test, kept)              # untouched by the gate
            po = np.load(run / 'logits' / ('iter1_%s.npy' % kind)).astype(np.float32).argmax(1)
            pr = np.load(ctrl / 'logits' / ('iter1_%s.npy' % kind)).astype(np.float32).argmax(1)
            acc = lambda p, i: float((p[i] == y[i]).mean()) if len(i) else np.nan
            rows.append({'dataset': ds, 'model': kind, 'seed': seed,
                         'n_test': len(test), 'n_kept': len(inn), 'n_not_kept': len(out),
                         'frac_test_leaked': len(inn) / len(test),
                         'kept_gain': acc(po, inn) - acc(pr, inn),
                         'not_kept_gain': acc(po, out) - acc(pr, out)})
    return pd.DataFrame(rows)

# ---- run it ----
pd.set_option('display.width', 210)
acc = final_accuracy()
eff = oracle_effect(acc)
print('ORACLE EFFECT — oracle minus oracle_random, paired by seed (test accuracy)')
print('shrink_cost = oracle_random - published, i.e. the cost of the size cut alone')
print(eff.round(4).to_string(index=False))

leak = leakage_decomposition()
summary = (leak.groupby(['dataset', 'model'])
               .agg(frac_test_leaked=('frac_test_leaked', 'mean'),
                    kept_gain=('kept_gain', 'mean'),
                    not_kept_gain=('not_kept_gain', 'mean'),
                    n_not_kept=('n_not_kept', 'mean'),
                    seeds=('seed', 'size')))
print()
print('LEAKAGE DECOMPOSITION — the oracle trains on correct labels for test nodes,')
print('so only `not_kept_gain` is free of leaked supervision (A15).')
print(summary.round(4).to_string())

### Reading it: a large headline, mostly artifact

**Headline.** The oracle beats the size-matched control on GNN test accuracy for every
dataset: pubmed +0.9pp (sd 0.002, seed-stable), cora +3.9pp, wisconsin +7.7pp,
washington +10.6pp, cornell +10.5pp. `shrink_cost` is near zero on the large datasets
(pubmed +0.0003, cora +0.0037), confirming the control is doing its job — dropping 5%
of pseudo-labels at random costs nothing, so the oracle's gain is about *which* labels
were dropped, not how many.

**But `frac_test_leaked` is 0.85–0.96 on cora, pubmed and most WebKB sets.** In the
oracle arm most of the test set is trained with its own correct label. Split the gain:

| dataset | model | kept-node gain | **not-kept gain** | n not-kept |
|---|---|---|---|---|
| citeseer | GNN | +0.124 | **−0.026** | 1962 |
| cora | GNN | +0.038 | +0.045 | 81 |
| pubmed | GNN | +0.010 | **−0.014** | 159 |
| pubmed | LM | +0.010 | +0.010 | 159 |
| WebKB | both | +0.017…+0.080 | −0.167…+0.187 | **1–9** |

Positive on the kept nodes everywhere, exactly as leakage predicts. On the residual —
the only leakage-free part — it is small, mixed in sign, and ≈0 or negative on the
three datasets with enough non-kept nodes to measure. Every WebKB residual rests on
1–9 nodes.

**So the ceiling as measured is inflated and should not be read as headroom for a
deployable gate.** As a formal upper bound on selection it still holds — nothing beats
selecting exactly the correct labels — but it is loose.

**The residual doubles as the cleanest harm test in the study.** On nodes where the
teacher is *wrong*, the oracle supplies no pseudo-label while the random control
supplies a wrong one. If wrong pseudo-labels harmed the student, the oracle would win
there. It does not, wherever n is large enough to tell — converging with the
preregistered null from a completely different direction.

A leakage-free version needs an evaluation split that never enters the pseudo-label
set. That is a different experiment, not a fix: GLEM pseudo-labels the whole unlabeled
set by construction.